# 06 · XGBoost, LightGBM y stacking sobre Wine Quality

**Módulo 4 · Sesión 11** — Boosting e interpretabilidad

## Objetivos

El notebook 05 construyó boosting a mano; este lo usa como se usa en la práctica —con
XGBoost y LightGBM— sobre los mismos datos, partición y pliegues de los notebooks 02 y 04,
para responder:

1. ¿Boosting **con valores por defecto** supera a Random Forest y Extra-Trees?
2. Cómo se elige el número de rondas con **early stopping**, en vez de adivinarlo.
3. Cuánto gana boosting **afinado con Optuna** (que se enseñó en el módulo 3; aquí solo se
   usa), y si esa ganancia sobrevive a la comparación pareada contra Extra-Trees.
4. Qué hace `scale_pos_weight` frente al desbalance, y si es distinto de mover el umbral.
5. Si **apilar** (stacking) la logística, Extra-Trees y LightGBM mejora al mejor de ellos.
6. El modelo final sobre el conjunto de prueba.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `xgboost`, `lightgbm`,
`optuna`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEMILLA = 42

## 1. Mismos datos, partición y pliegues que los notebooks 02 y 04

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv")
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)
vinos = vinos.drop_duplicates().reset_index(drop=True)
vinos["buena"] = (vinos["quality"] >= 7).astype(int)

X = vinos.drop(columns=["quality", "buena"])
y = vinos["buena"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)


def evaluar_cv(modelo, X=X_train, y=y_train, cv=cv):
    r = cross_validate(modelo, X, y, cv=cv, scoring=["average_precision", "roc_auc"])
    k = len(r["test_average_precision"])
    return pd.Series(
        {
            "AP": r["test_average_precision"].mean(),
            "ee AP": r["test_average_precision"].std(ddof=1) / np.sqrt(k),
            "AUC-ROC": r["test_roc_auc"].mean(),
            "ajuste (s)": r["fit_time"].mean(),
        }
    )

## 2. Boosting con valores por defecto, frente a los ensambles de la sesión 10

In [ ]:
modelos_defecto = {
    "Random Forest (S10)": RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
    "Extra-Trees (S10)": ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
    "AdaBoost (tocones)": AdaBoostClassifier(n_estimators=300, random_state=SEMILLA),
    "GradientBoosting (sklearn)": GradientBoostingClassifier(random_state=SEMILLA),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEMILLA),
    "XGBoost": XGBClassifier(random_state=SEMILLA, n_jobs=4, verbosity=0),
    "LightGBM": LGBMClassifier(random_state=SEMILLA, n_jobs=4, verbose=-1),
}
tabla_defecto = pd.DataFrame({n: evaluar_cv(m) for n, m in modelos_defecto.items()}).T
print(tabla_defecto.round(3).to_string())

Con los valores por defecto, **ningún boosting alcanza a Random Forest**, y Extra-Trees
les saca 0.03–0.05 de AP. AdaBoost con tocones queda al nivel de la logística. No es lo que
suele decirse de boosting, y por eso conviene medirlo: sus valores por defecto (100 rondas,
$\nu = 0.1$ o 0.3, profundidad 3 o 31 hojas) son un punto de partida, no una configuración.
Random Forest y Extra-Trees, en cambio, funcionan casi sin afinar — es su gran virtud
práctica.

## 3. Early stopping: cuántas rondas, sin adivinar

La receta del notebook 05: tasa baja, muchas rondas, y parar cuando la métrica sobre un
conjunto de validación deja de mejorar. LightGBM lo hace con `eval_set` y la retrollamada
`early_stopping`. Aquí se muestra sobre **un** pliegue; en la práctica se hace dentro de la
validación cruzada (sección 4).

In [ ]:
idx_tr, idx_va = next(cv.split(X_train, y_train))
X_tr, X_va = X_train.iloc[idx_tr], X_train.iloc[idx_va]
y_tr, y_va = y_train.iloc[idx_tr], y_train.iloc[idx_va]

lgbm_es = LGBMClassifier(n_estimators=3000, learning_rate=0.02, num_leaves=15, random_state=SEMILLA, n_jobs=4, verbose=-1)
lgbm_es.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_va, y_va)],
    eval_metric="average_precision",
    callbacks=[early_stopping(stopping_rounds=200, verbose=False)],
)
curva_tr = lgbm_es.evals_result_["training"]["average_precision"]
curva_va = lgbm_es.evals_result_["valid_1"]["average_precision"]

plt.figure(figsize=(7, 4))
plt.plot(curva_tr, label="entrenamiento")
plt.plot(curva_va, label="validación")
plt.axvline(lgbm_es.best_iteration_, color="gray", ls="--", label=f"mejor iteración: {lgbm_es.best_iteration_}")
plt.xlabel("Rondas")
plt.ylabel("AP")
plt.title("LightGBM (ν = 0.02, 15 hojas) con early stopping sobre un pliegue")
plt.legend()
plt.show()
print(f"AP de validación en la mejor iteración ({lgbm_es.best_iteration_}): {max(curva_va):.3f}; "
      f"se detuvo en la ronda {len(curva_va)}, 200 después del máximo")

La AP de entrenamiento sube hacia 1.0 —boosting memoriza si se le deja—, mientras que la
de validación alcanza su máximo y se estanca o baja. `best_iteration_` es la ronda que se
usaría. La curva es bastante plana alrededor del máximo, que es lo que la tasa de
aprendizaje baja compra (notebook 05, sección 2). El valor de AP de este pliegue (≈0.51)
es más bajo que el promedio de la CV porque es **un** pliegue —los cinco varían entre sí
más de lo que uno esperaría, y de ahí los errores estándar de ≈0.015 de todas las tablas—.

## 4. Afinar LightGBM con Optuna

Optuna ya se enseñó en `06-seleccion-hiperparametros.md`; aquí se **usa**. El espacio
incluye los hiperparámetros que de verdad mueven a un boosting: número de rondas y tasa,
tamaño del árbol (`num_leaves`, `min_child_samples`), submuestreo de filas y columnas, y
regularización $L_2$ de las hojas. El objetivo es la AP media de la misma CV de 5 pliegues.

In [ ]:
def objetivo(trial):
    parametros = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 4, 64, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    modelo = LGBMClassifier(**parametros, random_state=SEMILLA, n_jobs=4, verbose=-1)
    return cross_validate(modelo, X_train, y_train, cv=cv, scoring="average_precision")["test_score"].mean()


estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEMILLA, n_startup_trials=10))
estudio.optimize(objetivo, n_trials=40)

print(f"Mejor AP de CV: {estudio.best_value:.3f}")
print("Mejores hiperparámetros:")
for k, v in estudio.best_params.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

historial = np.maximum.accumulate([t.value for t in estudio.trials])
plt.figure(figsize=(7, 3.6))
plt.plot(historial, "o-", ms=3)
plt.axhline(tabla_defecto.loc["Extra-Trees (S10)", "AP"], color="gray", ls="--", label="Extra-Trees sin afinar")
plt.axhline(tabla_defecto.loc["LightGBM", "AP"], color="C3", ls=":", label="LightGBM por defecto")
plt.xlabel("Trial")
plt.ylabel("Mejor AP hasta el momento")
plt.legend()
plt.show()

Afinar sube a LightGBM de 0.54 a 0.57: recupera lo que perdía frente a Random Forest
(0.570), pero **no alcanza** a Extra-Trees sin afinar (0.589). La comparación no es del todo
justa —a Extra-Trees no se le buscó nada—, pero es la que importa en la práctica: cuánto
cuesta cada modelo llegar a un resultado dado. Aquí, 40 trials de Optuna no compraron lo
que Extra-Trees daba gratis.

### Comparación pareada: LightGBM afinado vs. Extra-Trees

Sobre los mismos 20 pliegues (5 × 4 repeticiones) del notebook 04. Una advertencia
honesta: los hiperparámetros de LightGBM se eligieron mirando estos mismos datos de
entrenamiento (la CV de 5 pliegues de arriba), así que su AP aquí tiene el sesgo optimista
que `06-seleccion-modelos-aplicado.ipynb` midió con CV anidada — pequeño, pero a favor de
LightGBM. El conjunto de prueba de la sección 7 no lo tiene.

In [ ]:
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEMILLA)
lgbm_afinado = LGBMClassifier(**estudio.best_params, subsample_freq=1, random_state=SEMILLA, n_jobs=4, verbose=-1)
extra_trees = ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1)


def ap_por_pliegue(modelo):
    return cross_validate(modelo, X_train, y_train, cv=cv_rep, scoring="average_precision")["test_score"]


ap_pliegues = {"LightGBM afinado": ap_por_pliegue(lgbm_afinado), "Extra-Trees": ap_por_pliegue(extra_trees)}


def comparar(a, b):
    d = ap_pliegues[a] - ap_pliegues[b]
    ee = d.std(ddof=1) / np.sqrt(len(d))
    return pd.Series({"AP de " + a: ap_pliegues[a].mean(), "AP de " + b: ap_pliegues[b].mean(),
                      "diferencia media": d.mean(), "ee": ee, "cociente": d.mean() / ee})


print(comparar("LightGBM afinado", "Extra-Trees").round(3).to_string())

Extra-Trees gana por 0.022 de AP con un error estándar de 0.004: **detectable** (cociente
5), aunque pequeño (un 4 % relativo), y con el sesgo de selección jugando a favor de
LightGBM. Es un resultado importante de recordar cuando se lee que "XGBoost/LightGBM gana
siempre": ganan a menudo, sobre todo con muchos datos y variables heterogéneas —los
ejercicios del módulo, sobre Adult Census, son ese caso—, pero sobre un dataset pequeño y
ruidoso un bosque aleatorio puede ganar sin que nadie lo afine. Ningún modelo es el mejor
en todos los datasets; la única forma de saberlo es medir.

## 5. `scale_pos_weight` frente al umbral

XGBoost y LightGBM tienen un hiperparámetro específico para el desbalance:
`scale_pos_weight` multiplica el gradiente de los positivos, con $n_{\text{neg}}/n_{\text{pos}}$
como valor habitual. Es el equivalente a `class_weight="balanced"`, y la pregunta es la
misma que en `02-clasificacion-aplicado.ipynb`: ¿cambia el orden de los vinos o solo
desplaza las probabilidades?

In [ ]:
peso = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = n_neg / n_pos = {peso:.2f}")


def mejor_f1(y, p):
    prec, rec, _ = precision_recall_curve(y, p)
    return (2 * prec * rec / np.maximum(prec + rec, 1e-12)).max()


filas = []
for nombre, spw in [("LightGBM afinado", 1.0), ("LightGBM afinado + scale_pos_weight", peso)]:
    m = LGBMClassifier(**estudio.best_params, subsample_freq=1, scale_pos_weight=spw, random_state=SEMILLA, n_jobs=4, verbose=-1)
    p = cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    filas.append({"modelo": nombre, "AUC-ROC": roc_auc_score(y_train, p), "AP": average_precision_score(y_train, p),
                  "recall en 0.5": recall_score(y_train, p >= 0.5), "F1 en 0.5": f1_score(y_train, p >= 0.5),
                  "mejor F1 (umbral libre)": mejor_f1(y_train, p), "P(buena) media": p.mean()})
print(pd.DataFrame(filas).set_index("modelo").round(3).to_string())

Mismo patrón que con la logística en el notebook 02: el recall en 0.5 sube, la escala de
las probabilidades cambia (la media predicha pasa de 0.15 a 0.19), y el AUC-ROC, la AP y el
mejor F1 alcanzable quedan **exactamente** donde estaban. `scale_pos_weight` es mover el
umbral con otro nombre. Si el umbral se va a elegir de todos modos (sección 7), no aporta
nada — y con valores grandes, deja las probabilidades sin relación con la frecuencia real
de la clase, que es un problema si alguien las va a leer como probabilidades.

## 6. Stacking: apilar tres modelos distintos

Un *stacking* entrena un **meta-modelo** sobre las predicciones de varios modelos base.
Para que el meta-modelo no aprenda de predicciones sobreajustadas, las predicciones de
entrenamiento se generan con validación cruzada interna (`cv=5` en `StackingClassifier`),
igual que `cross_val_predict`. La apuesta es que modelos con **sesgos distintos** —una
frontera lineal, un bosque aleatorio y un boosting— se equivoquen en sitios distintos.

In [ ]:
logistica = Pipeline([("escalar", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])
apilado = StackingClassifier(
    estimators=[("logistica", logistica), ("extra_trees", extra_trees), ("lightgbm", lgbm_afinado)],
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method="predict_proba",
    n_jobs=1,
)
ap_pliegues["Stacking"] = ap_por_pliegue(apilado)
print(comparar("Stacking", "Extra-Trees").round(3).to_string())

apilado.fit(X_train, y_train)
print("\nCoeficientes del meta-modelo (peso de cada modelo base en el log-momio final):")
print(pd.Series(apilado.final_estimator_.coef_[0], index=["logística", "extra_trees", "lightgbm"]).round(2).to_string())

El stacking gana a Extra-Trees por 0.006 de AP (cociente 2.5): detectable por los pelos,
un 1 % relativo — **detectable, no relevante**. Los coeficientes del meta-modelo cuentan
por qué: se apoya sobre todo en Extra-Trees, le da algo de peso a la logística (la única
con una frontera de forma distinta) y casi ignora a LightGBM, que se equivoca en los
mismos vinos que Extra-Trees. Es lo habitual: el stacking rinde cuando los modelos base
son **de verdad distintos** (por ejemplo, un modelo sobre texto y otro sobre variables
tabulares) y cuesta —tiempo de entrenamiento, y una capa más que explicar—.

## 7. Modelo final y conjunto de prueba

Extra-Trees es el mejor modelo en CV, no necesitó afinarse y se entrena en décimas de
segundo; el stacking le gana por un 1 % a cambio de tres modelos y una capa más. Se lleva
Extra-Trees al conjunto de prueba, con el umbral de mínimo costo del notebook 04 — y
también LightGBM afinado, para cerrar la comparación con un número sin sesgo de
selección.

In [ ]:
COSTO_FP, COSTO_FN = 1, 3
umbrales = np.round(np.linspace(0.05, 0.95, 91), 2)


def costo_total(y, p, umbral):
    pred = p >= umbral
    return COSTO_FP * np.sum(pred & (y == 0)) + COSTO_FN * np.sum(~pred & (y == 1))


resultado_prueba = []
for nombre, modelo in [("Extra-Trees", extra_trees), ("LightGBM afinado", lgbm_afinado)]:
    p_cv = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    umbral = umbrales[np.argmin([costo_total(y_train.to_numpy(), p_cv, u) for u in umbrales])]
    modelo.fit(X_train, y_train)
    p_test = modelo.predict_proba(X_test)[:, 1]
    pred_test = p_test >= umbral
    resultado_prueba.append({
        "modelo": nombre, "umbral (CV)": umbral,
        "AUC-ROC": roc_auc_score(y_test, p_test), "AP": average_precision_score(y_test, p_test),
        "precisión": precision_score(y_test, pred_test), "recall": recall_score(y_test, pred_test),
        "F1": f1_score(y_test, pred_test), "costo": costo_total(y_test.to_numpy(), p_test, umbral),
    })
print("Conjunto de prueba (1064 vinos):")
print(pd.DataFrame(resultado_prueba).set_index("modelo").round(3).to_string())
print("\nReferencia (notebook 02, logística): AP 0.481, F1 0.521, costo 383.")

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Boosting por defecto gana a los bosques? | No: AP 0.54–0.56 frente a 0.57 (RF) y 0.59 (Extra-Trees) |
| ¿Cuántas rondas? | Las que diga el early stopping sobre validación, con $\nu$ baja; la curva es plana cerca del máximo |
| ¿Y afinado con Optuna? | Sube a 0.57: alcanza a Random Forest, pero Extra-Trees sin afinar sigue 0.022 ± 0.004 por encima |
| `scale_pos_weight` | Sube el recall en 0.5 y reescala las probabilidades; AUC, AP y mejor F1 no cambian: es mover el umbral |
| Stacking | +0.006 ± 0.003 sobre Extra-Trees: detectable, no relevante; el meta-modelo casi ignora a LightGBM |
| Modelo final | Extra-Trees, con el umbral elegido en CV: AP 0.62 en prueba frente a 0.58 de LightGBM y 0.48 de la logística |

Queda la pregunta que ningún ensamble responde por sí solo: **qué variables** usan, y cómo.
Es el tema del notebook 07.